# UIT Chatbot — Fine-tuning Qwen2.5-7B với QLoRA (Unsloth)

**Mục tiêu:** Fine-tune Qwen2.5-7B trên dataset Q&A đào tạo UIT bằng QLoRA trên Google Colab T4.

**Checklist trước khi chạy:**
- [ ] Chọn Runtime → Change runtime type → T4 GPU
- [ ] Upload `train.jsonl` và `eval.jsonl` lên Google Drive
- [ ] Cập nhật `GDRIVE_DATA_PATH` bên dưới

In [ ]:
# Cell 1: Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q xformers trl peft bitsandbytes accelerate datasets

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Cell 2: Mount Google Drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# ---- CONFIGURE THIS PATH ----
GDRIVE_DATA_PATH = '/content/drive/MyDrive/uit-chatbot-data'
# -----------------------------

import os
train_path = os.path.join(GDRIVE_DATA_PATH, 'train.jsonl')
eval_path  = os.path.join(GDRIVE_DATA_PATH, 'eval.jsonl')

assert os.path.exists(train_path), f'train.jsonl not found at {train_path}'
assert os.path.exists(eval_path),  f'eval.jsonl not found at {eval_path}'

from datasets import load_dataset
dataset = load_dataset('json', data_files={'train': train_path, 'eval': eval_path})
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['eval'])}")

In [ ]:
# Cell 3: Load Qwen2.5-7B with Unsloth (4-bit QLoRA)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None          # Auto-detect (bfloat16 on A100, float16 on T4)
LOAD_IN_4BIT = True   # QLoRA — required for T4 16GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Model loaded successfully')

In [ ]:
# Cell 4: Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Saves ~20% VRAM
    random_state=42,
)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M | Trainable: {train_params/1e6:.1f}M ({100*train_params/total_params:.1f}%)')

In [ ]:
# Cell 5: Format dataset with ChatML template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='chatml')

def format_sample(sample):
    """Convert messages list to tokenizer's chat template string."""
    return {'text': tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )}

train_dataset = dataset['train'].map(format_sample, batched=False)
eval_dataset  = dataset['eval'].map(format_sample, batched=False)

print('Sample formatted:')
print(train_dataset[0]['text'][:500])

In [ ]:
# Cell 6: Training
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

CHECKPOINT_DIR = '/content/drive/MyDrive/uit-chatbot-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,    # Effective batch = 8
        num_train_epochs=1,
        warmup_steps=50,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=100,
        save_steps=100,                   # Checkpoint every 100 steps
        save_total_limit=3,
        output_dir=CHECKPOINT_DIR,
        optim='adamw_8bit',               # 8-bit optimizer for memory
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Training complete — {trainer_stats.metrics}')

In [ ]:
# Cell 7: Inference test before export
FastLanguageModel.for_inference(model)

test_messages = [
    {'role': 'system', 'content': 'Bạn là tư vấn viên thông tin đào tạo của UIT. Hãy trả lời chính xác và thân thiện.'},
    {'role': 'user',   'content': 'Làm thế nào để đăng ký học phần?'},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Cell 8: Export to GGUF (q4_k_m) for Ollama inference
GGUF_OUTPUT_DIR = '/content/drive/MyDrive/uit-chatbot-gguf'
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

# Save merged model then convert to GGUF
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method='q4_k_m',  # ~5GB, best quality/size tradeoff
)

print(f'GGUF model saved to {GGUF_OUTPUT_DIR}')
print('Next steps:')
print('  1. Download the .gguf file from Google Drive')
print('  2. Place in models/ directory')
print('  3. Create Ollama modelfile: ollama create uit-chatbot -f models/Modelfile')
print('  4. Run: ollama serve')